In [ ]:
# Based on the complecity of thr database

In [ ]:
#Create a sql server get the details 
#Check number of rows or the details of the tables 
#If more that some number give only the sql tables 
#Get the tables which are relavant to the us

In [ ]:
!pip install psycopg2

In [ ]:
import psycopg2
import os
from dotenv import load_dotenv
load_dotenv()
conn = psycopg2.connect(os.environ['DATABASE_URL'])

In [ ]:
def get_database_schema(conn : psycopg2.extensions.connection) -> dict:
    """
    Retrieve the database schema details from the PostgreSQL database.

    Args:
        conn (psycopg2.extensions.connection): A connection object to the PostgreSQL database

    Returns:
        dict: A dictionary containing the schema details for each table in the database.
    """
    with conn.cursor() as cursor:
        cursor.execute("""
                    SELECT
                        table_name,
                        column_name,
                        data_type,
                        is_nullable,
                        column_default|
                    FROM information_schema.columns
                    WHERE table_schema = 'public'
                    ORDER BY table_name, ordinal_position;
                    """)
        tables = cursor.fetchall()
        print("Tables in the database:")
        tables_details = {}
        for table in tables:
            table_name, column_name, data_type, is_nullable, column_default = table
            if table_name not in tables_details:
                tables_details[table_name] = []
            tables_details[table_name].append({
                "column_name": column_name,
                "column_default": column_default
            })
    return tables_details

In [ ]:
!pip install langchain-mistralai

# LLM model

In [ ]:
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatMistralAI(    
    model="mistral-small-2506",
    temperature=0,
    max_retries=2
)

In [ ]:
#Simple agent to parse excel data and get the relevant tables and columns 

# Cleaning and updating the database

In [ ]:
import pandas as pd

def convert_numeric_columns(df):
    for col in df.columns:
        # try convert to numeric
        converted = pd.to_numeric(df[col], errors='ignore')

        # if conversion successful (not object anymore)
        if pd.api.types.is_numeric_dtype(converted):
            # check if it's integer-like
            if (converted.dropna() % 1 == 0).all():
                df[col] = converted.astype("Int64")  # nullable int
            else:
                df[col] = converted.astype(float)

    return df
def safe_fillna(df):
    for col in df.columns:
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = df[col].astype("Int64")  # keep nullable int
        elif pd.api.types.is_float_dtype(df[col]):
            df[col] = df[col].astype(float)
        else:
            df[col] = df[col].astype(str).fillna("")

    return df

In [ ]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
import re
import os

from dotenv import load_dotenv
load_dotenv()

# -------------------------
# DB CONNECTION
# -------------------------
def get_conn():
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        port=5432,
        user='layerbase',
        password=os.getenv("DB_PASSWORD"),
        dbname='duckdb_test'
    )


In [ ]:


# -------------------------
# CLEAN COLUMN NAMES
# -------------------------
def clean_columns(columns):
    cleaned = []
    for c in columns:
        c = str(c).strip().lower()
        c = re.sub(r"[^a-zA-Z0-9_]", "_", c)
        c = re.sub(r"_+", "_", c).strip("_")
        cleaned.append(c or "col")
    return cleaned


# -------------------------
# FIX: convert string columns that are actually ints/floats
# -------------------------
def coerce_numeric_strings(df: pd.DataFrame) -> pd.DataFrame:
    """
    For every object/string column, check if every non-null value looks
    like an integer (allowing stray whitespace, thousands separators,
    or a trailing '.0' from Excel float formatting). If so, cast the
    whole column to nullable Int64. If values are numeric but have real
    decimals, cast to float instead. Genuinely non-numeric text columns
    are left untouched.
    """
    int_pattern = re.compile(r"^-?\d+$")

    for col in df.columns:
        if df[col].dtype != object:
            continue

        series = df[col]
        non_null = series.dropna()
        if len(non_null) == 0:
            continue

        def normalize(v):
            if not isinstance(v, str):
                return v
            v = v.strip().replace(",", "")
            if re.match(r"^-?\d+\.0$", v):
                v = v[:-2]
            return v

        normalized = non_null.map(normalize)

        all_int = normalized.map(lambda v: bool(int_pattern.match(str(v)))).all()
        if all_int:
            df[col] = pd.to_numeric(series.map(normalize), errors="coerce").astype("Int64")
            continue

        def is_float(v):
            try:
                float(v)
                return True
            except (TypeError, ValueError):
                return False

        if normalized.map(is_float).all():
            df[col] = pd.to_numeric(series.map(normalize), errors="coerce")

    return df


# -------------------------
# INFER SQL TYPES
# -------------------------
def infer_sql_type(series):
    if pd.api.types.is_integer_dtype(series):
        return "BIGINT"
    elif pd.api.types.is_float_dtype(series):
        return "DOUBLE PRECISION"
    elif pd.api.types.is_bool_dtype(series):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(series):
        return "TIMESTAMP"
    else:
        return "TEXT"


# -------------------------
# CREATE TABLE
# -------------------------
def create_table(cur, table_name, df):
    cols = [f'"{col}" {infer_sql_type(df[col])}' for col in df.columns]
    col_sql = ", ".join(cols)
    query = f'CREATE TABLE IF NOT EXISTS "{table_name}" ({col_sql})'
    cur.execute(query)


# -------------------------
# BULK INSERT
# -------------------------
def insert_data(cur, table_name, df):
    cols = ",".join([f'"{c}"' for c in df.columns])
    # convert pandas NA/NaN/NaT -> None, and numpy scalar types -> native python
    values = [
        tuple(None if pd.isna(v) else (v.item() if hasattr(v, "item") else v) for v in row)
        for row in df.to_numpy()
    ]
    query = f'INSERT INTO "{table_name}" ({cols}) VALUES %s'
    execute_values(cur, query, values)


# -------------------------
# MAIN PIPELINE
# -------------------------
def load_excel(file_path, table_name=None):
    if table_name is None:
        table_name = os.path.splitext(os.path.basename(file_path))[0]
        table_name = re.sub(r"[^a-zA-Z0-9_]", "_", table_name)
    
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".xlsx":
        df = pd.read_excel(file_path, engine="openpyxl")
    elif ext == ".xls":
        df = pd.read_excel(file_path, engine="xlrd")
    else:
        raise ValueError("Unsupported file format")
    df = pd.read_excel(file_path, engine="openpyxl")
    df.columns = clean_columns(df.columns)

    # >>> key fix: strings that are really ints get cast properly <
    df = coerce_numeric_strings(df)

    df = df.where(pd.notnull(df), None)

    conn = get_conn()
    cur = conn.cursor()

    try:
        create_table(cur, table_name, df)
        insert_data(cur, table_name, df)
        conn.commit()
        print(f"✅ Loaded {len(df)} rows into '{table_name}'")
    except Exception as e:
        conn.rollback()
        print("❌ Error:", str(e))
    finally:
        cur.close()
        conn.close()


if __name__ == "__main__":
    load_excel("gujrat.xlsx" , table_name="product_details")

In [ ]:
!pip install xlrd

In [ ]:
conn = get_conn()
with conn.cursor() as cursor:
    cursor.execute("describe mix;")
    tables = cursor.fetchall()
    print("Tables in the database:")
    for table in tables:
        print(table[:2])
    current_table = "mix"
    cursor.execute(f"describe {current_table};")
    print(len(tables), "tables found in the database.")

    print(len(tables[0]), "columns found in the 'mix' table.")

# STATE

In [ ]:
from langchain_core.runnables import RunnableConfig

In [ ]:
from typing import TypedDict

class SqlQueryRetrialState(TypedDict):
    user_question : str
    tables_available : list 
    table_details : dict 
    required_tables : list
    query : str 
    verified : bool = False
    retry_no : int = 0 
    last_sql: str
    last_error: str
    confidence : float = 0.0
    last_suggetion : str = ""
    last_issues : list[str] = ""
    final_answer : str = ""
    result : list


# Structured Output Classes

In [ ]:
from pydantic import BaseModel, Field

class RequiredTables(BaseModel):
    table_name : list[str] = Field(description="List of required table names for the SQL query")

class QueryResult(BaseModel):
    query : str = Field(description="The SQL query generated by the agent")
    confidence : float = Field(description="Confidence score of the generated SQL query")


class CriticResult(BaseModel):
    approved: bool = Field(description="Whether the SQL query is correct and safe to execute.")

    confidence: float = Field(ge=0.0,le=1.0,description="Confidence score between 0.0 and 1.0.")

    issues: list[str] = Field(default_factory=list,description="List of problems found in the SQL query.")

    suggestions: list[str] = Field(default_factory=list,description="Concrete suggestions to fix the issues.")

    summary: str = Field(description="Short overall assessment of the SQL query.")

class FinalAnswer(BaseModel):
    answer: str = Field(description="The final answer to the user's question, based on the SQL query results or other reasoning.")


class ChitChatResult(BaseModel):
    result : str = Field(description="Response generate by the chitchat agent")    

# System Prompt

In [ ]:
get_required_table_prompt = """" 
You are a SQL query agent. You are given a question and a database schema. Your task is to identify the relevant tables from the schema that are needed to answer the question.

{user_question}
The avalible tables are 
{tables_available}
Note - Only return the table names that are explicitly mentioned in the question or are necessary to answer the question.
Dont provide any other information or explanation. 
Dont invent table names that are not present in the schema.
Make sure to ingore any tables that are not relevant to the question and default tables.
"""



generate_sql_query_prompt = """
You are an expert DuckDB SQL generator.
Generate a single read-only DuckDB SQL query that answers the user's question using ONLY the provided schema.
User Question:
{user_question}
Relevant Tables:
{relevant_tables}
Schema:
{table_details}
Rules:
- Generate exactly ONE SQL statement.
- Only SELECT statements are allowed. A WITH clause (CTE) is allowed.
- Never generate INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, TRUNCATE, COPY, PRAGMA, ATTACH, DETACH, CALL, or any non-read operation.
- Use ONLY the tables and columns present in the provided schema.
- Never invent tables or columns.
- Use the exact table and column names from the schema.
- Quote identifiers containing spaces or special characters using double quotes.
- Prefer explicit column names over SELECT * unless the user explicitly requests all columns.
- Include only the columns required to answer the question.
- Use appropriate WHERE, GROUP BY, HAVING, ORDER BY and LIMIT clauses when needed.
- When calculating totals, averages, counts or other metrics, use SQL aggregation functions.
- When grouping, include all non-aggregated selected columns in the GROUP BY clause.
- Use DuckDB-compatible SQL only.
- Prefer native DuckDB functions.
- Do not use MySQL-, SQL Server-, Oracle-, or PostgreSQL-specific syntax unless it is also supported by DuckDB.
- Assume numeric columns are already stored as numeric types unless the schema indicates otherwise.
- For text filtering, use case-insensitive matching with ILIKE when the user refers to names or text values without specifying an exact match.
- If the question requests the "top", "highest", "lowest", "most", or "least", include an appropriate ORDER BY and LIMIT clause.
- If the question cannot be answered using the supplied schema, generate a query that clearly indicates the schema is insufficient rather than inventing columns or tables.
Use previous query or error or issues to improve the next query if needed.
Previous QUERY -  {prev_query} , error - {prev_error}, issues - {prev_issues}, suggestions - {prev_suggetion}

Generate only the SQL query.
"""


critic_check_prompt = """
You are an expert SQL reviewer specializing in DuckDB.

Review the generated SQL query against the user's question and the provided database schema.

User Question:
{user_question}

Database Schema:
{table_details}

Generated SQL:
{sql_query}

Evaluate the query using the following checklist:

1. Correctness
- Does the query answer the user's question?
- Does it return the requested information?
- Are the correct tables and columns used?
- Has the query misunderstood the user's intent?

2. Schema Validation
- Every referenced table must exist.
- Every referenced column must exist.
- Do not allow invented tables or columns.

3. SQL Validity
- The query must be valid DuckDB SQL.
- Only a single read-only SELECT statement is allowed.
- A WITH clause (CTE) is allowed.
- Reject INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, TRUNCATE, COPY, PRAGMA, ATTACH, DETACH, CALL, or multiple SQL statements.

4. Aggregation
- Verify GROUP BY usage.
- Verify aggregate functions are used correctly.
- Detect missing GROUP BY columns.
- Detect unnecessary aggregation.

5. Filtering
- Verify WHERE conditions match the user's intent.
- Verify joins and predicates are logically correct.

6. Performance
- Detect unnecessary SELECT *.
- Detect unnecessary joins.
- Detect redundant subqueries.
- Detect unnecessary DISTINCT.
- Recommend LIMIT only when appropriate.

7. DuckDB Compatibility
- Reject database-specific syntax that DuckDB does not support.
- Prefer native DuckDB functions.

8. Safety
- Ensure the query is read-only.
- Ensure no SQL injection patterns or dynamic SQL are present.

Based on your review:

- If the query is correct, efficient, valid, and answers the user's question, approve it.
- Otherwise, explain every issue found and describe how the query should be corrected.
- Do not invent schema objects.
- Base every decision solely on the supplied schema and user question.
"""


final_answer_prompt = """
You are an expert data analyst.

A SQL query has already been generated and executed successfully.
Your task is to answer the user's question using ONLY the query results provided.

User Question:
{user_question}

Query Results:
{query_results}

Instructions:
- Answer the user's question directly and concisely.
- Base your answer ONLY on the query results.
- Do not infer or invent information that is not present in the results.
- Do not mention SQL, databases, tables, columns, or query execution.
- If the result contains aggregated values, explain them naturally.
- If multiple rows are returned, summarize them clearly. Use bullet points only if they improve readability.
- If the result set is empty, state that no matching records were found.
- Preserve numeric precision where appropriate.
- Preserve dates and timestamps without changing their meaning.
- Do not include reasoning, assumptions, or internal analysis.
- Return only the final answer.
"""

In [ ]:
# Helper functions

def execute_sql_query(conn , query):
    """
    Execute a SQL query and return the results.

    Args:
        conn (psycopg2.extensions.connection): A connection object to the PostgreSQL database.
        query (str): The SQL query to execute.
    """

    with conn.cursor() as cursor:
        try:
            cursor.execute(query)
            if cursor.description:  
                results = cursor.fetchall()
                return results
            else:
                conn.rollback()# Commit if it's an INSERT/UPDATE/DELETE
                raise PermissionError("SQL execution is not allowed.")
        except Exception as e:
            print(f"Error executing query: {e}")
            conn.rollback()
            raise RuntimeError(
                f"Failed to execute SQL.\nQuery:\n{query}\nError: {e}"
            ) from e
#Tools for agents 
def get_database_schema(conn) -> list[str]:
    """
    Retrieve the database schema details from the PostgreSQL database.

    Args:
        conn (psycopg2.extensions.connection): A connection object to the PostgreSQL database
    returns 
        dict: A dictionary containing the schema details for each table in the database.
    """
    query = "SELECT table_name FROM duckdb_tables() WHERE schema_name = 'main' ORDER BY table_name;"
    result =  execute_sql_query(conn, query)
    return [row[0] for row in result] if result else []


def get_table_details_by_name(conn , table_name: str) -> dict:
    """
    Retrieve the details of a specific table from the PostgreSQL database.

    Args:
        conn (psycopg2.extensions.connection): A connection object to the PostgreSQL database
        table_name (str): The name of the table to retrieve details for.
    """
    query = f"DESCRIBE {table_name};"
    result = execute_sql_query(conn, query)
    if result:
        return {
            "table_name": table_name,
            "columns": [{ row[0], row[1]} for row in result]
        }
    else:
        return {"table_name": table_name, "columns": []}

In [ ]:
#Create a agent which can excecute the sql queries and get the result 


#START

# Node to get the input start node 
# Node to get the details of the tables 
def get_database_details(state : SqlQueryRetrialState , config : RunnableConfig) -> dict:
    print("config" , config)
    return {
        'table_details' : get_database_schema(config.get("configurable", {}).get("conn")),
    }
def get_required_tables(state: SqlQueryRetrialState) -> dict:
    """Ask the LLM which tables are needed to answer the question."""
    llm_structured = llm.with_structured_output(RequiredTables)
 
    prompt = get_required_table_prompt.format(
        user_question=state['user_question'],
        tables_available=state['table_details'],
    )
    response = llm_structured.invoke(prompt)
 
    return {
        'required_tables': response.model_dump()['table_name'],
    }
 
# Node to get the details of the relevant tables
def get_table_details(state: SqlQueryRetrialState , config : RunnableConfig) :
    table_details = {}
    print(table_details)
    for table in state['required_tables']:
        table_details[table] = get_table_details_by_name(
                                                        config.get("configurable", {}).get("conn"), 
                                                        table).get("columns")
        print(f"Got details of {table}")
    print(table_details)
    return {
        'table_details' : table_details
    }
    
def chitchat(state : SqlQueryRetrialState):
    pass
    

# Node to get the query 
def get_query(state: SqlQueryRetrialState):
    


    llm_structed = llm.with_structured_output(QueryResult)
    prev_query = state.get('last_sql', "")
    prev_error = state.get('last_error', "")
    prev_suggestion = state.get('last_suggetion', "")
    prev_issues = state.get('last_issues', "")
        
    prompt = generate_sql_query_prompt.format(
        user_question=state['user_question'], 
        relevant_tables=state['required_tables'], 
        table_details=state['table_details'],
        prev_query = prev_query ,
        prev_error = prev_error,
        prev_suggetion = prev_suggestion,
        prev_issues = prev_issues
    )
    response = llm_structed.invoke(prompt)
    return {
        'query' : response.model_dump()['query'],
        'confidence' : response.model_dump()['confidence']
    }

# Node to make sure the query is safe execute 
def critic(state: SqlQueryRetrialState) -> dict:
    """
    Validate the generated query before it's executed.
 
    retry_no is incremented HERE (inside a real node) instead of inside the
    conditional-edge function. Conditional-edge functions in LangGraph only
    return a routing string - any state mutation inside them is silently
    discarded, so incrementing retry_no there never actually persisted.
    """
    llm_structured = llm.with_structured_output(CriticResult)
    prompt = critic_check_prompt.format(
        user_question=state['user_question'],
        table_details=state['table_details'],
        sql_query=state['query'],
    )
 
    response = None
    for attempt in range(3):  # bounded instead of the original `while True`
        try:
            response = llm_structured.invoke(prompt)
            if response is not None:
                break
        except Exception as e:
            print(f"critic invoke failed (attempt {attempt + 1}/3): {e}")
    
 
    retry_no = state.get('retry_no', 0) + 1
 
    if response is None:
        # Structured output kept failing - fail safe instead of hanging forever.
        return {
            'verified': False,
            'confidence': 0,
            'last_issues': "Critic failed to produce a response after 3 attempts.",
            'last_suggetion': "Retry query generation.",
            'summary': "",
            'retry_no': retry_no,
        }
 
    result = response.model_dump()
    return {
        'verified': result['approved'],
        'confidence': result['confidence'],
        'last_issues': result['issues'],
        'last_suggetion': result['suggestions'],
        'summary': result['summary'],
        'retry_no': retry_no,
    }
 

def retry_query(state: SqlQueryRetrialState):
    state['retry_no'] +=1 
    if state['verified'] :
        return 'excute_query'
    else:
        if state['retry_no'] > 3:
            return 'format_result'
        else:
            return "query_generator"
# Node executes the query and get the result 
def execute_query(state: SqlQueryRetrialState , config : RunnableConfig):
    try :
        result = execute_sql_query(
        config.get("configurable", {}).get("conn"),
        state['query']
        )
    except Exception as e:
        print(e)
        return {
            'result' : None,
            'last_error' : str(e),
            'verified' : False
        }
    return {'result': result}

def is_query_successful(state: SqlQueryRetrialState) -> str:
    """
    Conditional edge after `execute_query`.
 
    Original code had this inverted: a successful result routed back to
    query_generator and a failed one went to format_result. Fixed so a
    successful result moves on to summarization, and a failure retries
    (bounded by retry_no) before giving up.
    """
    if state['result'] is not None:
        return 'format_result'
    if state.get('retry_no', 0) > 3:
        return 'format_result'
    return 'query_generator'
# Node checks weather the result  and provides in the required format
def format_result(state: SqlQueryRetrialState):
    llm_structed = llm.with_structured_output(FinalAnswer)
    prompt = final_answer_prompt.format(
        user_question=state['user_question'],
        query_results=state['result']
    )
    response = llm_structed.invoke(prompt)
    return {
        'final_answer' : response.model_dump()['answer']
    }


# END


In [ ]:
from langgraph.graph import StateGraph , START ,END
from langgraph.checkpoint.memory import InMemorySaver


builder = StateGraph(SqlQueryRetrialState)
builder.add_node('database_details' , get_database_details)
builder.add_node('required_tables' , get_required_tables)
builder.add_node('table_details' , get_table_details)
builder.add_node('query_generator' , get_query)
builder.add_node('critic' , critic)
builder.add_node('excute_query' , execute_query)
builder.add_node('format_result' , format_result)

builder.add_edge(START , 'database_details')
builder.add_edge('database_details' , 'required_tables' )
builder.add_edge('required_tables' , 'table_details' )
builder.add_edge('table_details' , 'query_generator')
builder.add_edge('query_generator' , 'critic')
builder.add_edge('format_result' , END)
builder.add_conditional_edges('critic' ,
                              retry_query, 
                              {
                                 "excute_query" : "excute_query",
                                 "query_generator" : "query_generator",
                                 "format_result" : "format_result"
                              })

builder.add_conditional_edges('excute_query' , 
                              is_query_successful ,
                              {
                                  "query_generator" : "query_generator",
                                  "format_result" : "format_result"
                              }
                              )

In [ ]:
config = {
    "configurable" : {
        "thread_id" :  "1" ,
        "conn" : get_conn()
    }
}

In [ ]:
agent = builder.compile(checkpointer=InMemorySaver())

In [ ]:
agent

In [ ]:
from pprint import pprint

for state in agent.stream(
    {
        "user_question": "Get the details of all top 5 based on the bill amount", 
        'retry_no' : 0
    },
    config=config,
    stream_mode="values",
):
    pprint(state)

In [ ]:
state = agent.get_state(config)

In [ ]:
state.next

In [ ]:
state.next

In [ ]:
print(state.values['result'])

In [ ]:
print(state.values['final_answer'])

In [ ]:
from pprint import pprint

for state in agent.stream(
    {
        "user_question": "Wht are there GST number", 
        'retry_no' : 0
    },
    config=config,
    stream_mode="values",
):
    pprint(state)

In [ ]:
new_state = agent.get_state(config)

In [ ]:
new_state.next


In [ ]:
new_state

In [ ]:
new_state.values['final_answer']

In [ ]:
new_state.values['result']

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)
import os
print(os.getcwd())

In [ ]:
from langsmith import Client
client = Client()
project = client.list_projects(limit=1)

In [ ]:
project = list(project)

In [ ]:
from langchain_mistralai import ChatMistralAI
from langsmith import traceable

@traceable(
    run_type="llm",
    metadata={"ls_provider": "mistral", "ls_model_name": "mistral-medium-latest"},
)
def query_mistral(prompt: str):
    response = client.chat.complete(
        model="mistral-medium-latest",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message

In [ ]:
query_mistral("What is the best thing in life")